In [18]:
# Core
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# Statsmodels
import statsmodels.api as sm


from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer, make_column_selector as selector
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor


In [10]:

# EDA from week2 Mod B
df_diabetes = pd.read_csv("diabetes_binary_health_indicators_BRFSS2015.csv")
df_ckd = pd.read_csv("Chronic_Kidney_Dsease_data.csv")
df_hypertension = pd.read_csv("hypertension_dataset.csv")
df_alzheimers = pd.read_csv("alzheimers_disease_data.csv")

# Generate basic summaries
diabetes_desc = df_diabetes.describe(include='all')
ckd_desc = df_ckd.describe(include='all')
hypertension_desc = df_hypertension.describe(include='all')
alzheimers_desc = df_alzheimers.describe(include='all')

# Check for duplicates
diabetes_duplicates = df_diabetes.duplicated().sum()
ckd_duplicates = df_ckd.duplicated().sum()
alzheimers_duplicates = df_alzheimers.duplicated().sum()
_duplicates = df_hypertension.duplicated().sum()
# Check for nulls
diabetes_nulls = df_diabetes.isnull().sum()
ckd_nulls = df_ckd.isnull().sum()
hypertension_nulls = df_hypertension.isnull().sum()
alzheimers__nulls = df_alzheimers.isnull().sum()

# Display descriptive summaries

print("\n===alzheimers-Summary ===")
print(ckd_desc)
# Return duplicates and total null values per dataset
print("\n=== Duplicates in alzheimers_duplicates Dataset ===")
print(f"alzheimers: {alzheimers_duplicates}")

# Return nulls
print("\n=== Total Null Values in alzheimers Dataset ===")
print(f"Diabealzheimerstes: {alzheimers__nulls.sum()}")


#Diabetes Dataset
#Duplicates: 24,206 rows — significant duplication, should be reviewed or removed
#Missing values: None
#Usability: Usable after deduplication

#Chronic Kidney Disease (CKD) Dataset
#Duplicates: None
#Missing values: None\
#Usability: Clean and ready for analysis

#Hypertension Dataset
#Duplicates: None
#Missing values: None
#Usability: Ready to use

# Steps to clean up Diabetes Dataset 
#Remove duplicates from the diabetes dataset: df_diabetes.drop_duplicates(inplace=True)
#Consider class balance checks (e.g., ratio of positive to negative labels)
#Identify categorical features and apply encoding (pd.get_dummies or OrdinalEncoder)
#Explore mode, median, and outliers for inconsistent data (e.g., age = 0)

#1 Remove Duplicates
df_diabetes.drop_duplicates(inplace=True)
#2 Handling any missing values
df_diabetes.fillna(df_diabetes.median(), inplace=True)  # For numeric columns
df_diabetes.fillna("Unknown", inplace=True)    # For categorical columns
#3Check for Inconsistencies
  #Negative ages or values outside expected range
  #Incorrect data types (e.g., numeric coded as string)
#4 Check for Class Imbalance

print(df_diabetes['Diabetes_binary'].value_counts(normalize=True))
print(df_hypertension['Hypertension'].value_counts(normalize=True))
#print(df_ckd['classification'].value_counts(normalize=True)) 

# Encode Categorical Variables
# One-hot encoding (for logistic regression, tree-based models)
df = pd.get_dummies(df_diabetes, drop_first=True)

# Or ordinal encoding if there is a natural order
#print("Arun")
#print(df_ckd.columns)
categorical_cols = df_ckd.select_dtypes(include=['object', 'category']).columns.tolist()
#print("Categorical columns:", categorical_cols)

from sklearn.preprocessing import OrdinalEncoder
encoder = OrdinalEncoder()
df_ckd[['DoctorInCharge']] = encoder.fit_transform(df_ckd[['DoctorInCharge']])




===alzheimers-Summary ===
          PatientID          Age       Gender   Ethnicity  \
count   1659.000000  1659.000000  1659.000000  1659.00000   
unique          NaN          NaN          NaN         NaN   
top             NaN          NaN          NaN         NaN   
freq            NaN          NaN          NaN         NaN   
mean     830.000000    54.441230     0.515371     0.71308   
std      479.056364    20.549757     0.499914     1.00043   
min        1.000000    20.000000     0.000000     0.00000   
25%      415.500000    36.000000     0.000000     0.00000   
50%      830.000000    54.000000     1.000000     0.00000   
75%     1244.500000    72.000000     1.000000     1.00000   
max     1659.000000    90.000000     1.000000     3.00000   

        SocioeconomicStatus  EducationLevel          BMI      Smoking  \
count           1659.000000     1659.000000  1659.000000  1659.000000   
unique                  NaN             NaN          NaN          NaN   
top                  

In [20]:
# ===== 1) Load & target
df=df_alzheimers
target_col = "MMSE"  


X = df.drop(columns=[target_col] + drop_cols, errors="ignore")
y = df[target_col]

numeric_features = selector(dtype_include=np.number)(X)
categorical_features = selector(dtype_include=object)(X)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)



In [23]:
rop_cols = [c for c in ["PatientID", "DoctorInCharge"] if c in df.columns]
X = df.drop(columns=[target_col] + drop_cols, errors="ignore")
y = df[target_col]

numeric_features = selector(dtype_include=np.number)(X)
categorical_features = selector(dtype_include=object)(X)

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# ---- Decision Tree (small grid) ----
dt_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", DecisionTreeRegressor(random_state=42))
])

dt_grid = {
    "model__max_depth": [None, 4, 6, 8, 10, 14],
    "model__min_samples_split": [2, 5, 10],
    "model__min_samples_leaf": [1, 2, 4]
}

dt_cv = GridSearchCV(dt_pipe, dt_grid, cv=3, n_jobs=1, scoring="neg_mean_squared_error")
dt_cv.fit(X_train, y_train)
dt_best = dt_cv.best_estimator_

# ---- Random Forest (small grid) ----
rf_pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", RandomForestRegressor(random_state=42, n_jobs=1))
])

rf_grid = {
    "model__n_estimators": [150, 250],
    "model__max_depth": [None, 10, 16],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2],
    "model__max_features": ["sqrt", "log2"]
}

rf_cv = GridSearchCV(rf_pipe, rf_grid, cv=3, n_jobs=1, scoring="neg_mean_squared_error")
rf_cv.fit(X_train, y_train)
rf_best = rf_cv.best_estimator_

def metrics(model, X_tr, X_te, y_tr, y_te):
    yhat_tr = model.predict(X_tr)
    yhat_te = model.predict(X_te)
    mse_tr = mean_squared_error(y_tr, yhat_tr); rmse_tr = np.sqrt(mse_tr)
    mse_te = mean_squared_error(y_te, yhat_te); rmse_te = np.sqrt(mse_te)
    mae_tr = mean_absolute_error(y_tr, yhat_tr)
    mae_te = mean_absolute_error(y_te, yhat_te)
    r2_tr = r2_score(y_tr, yhat_tr)
    r2_te = r2_score(y_te, yhat_te)
    return rmse_tr, mae_tr, r2_tr, rmse_te, mae_te, r2_te

rows = []
for name, model, cvres in [
    ("DecisionTree", dt_best, dt_cv),
    ("RandomForest", rf_best, rf_cv)
]:
    rmse_tr, mae_tr, r2_tr, rmse_te, mae_te, r2_te = metrics(model, X_train, X_test, y_train, y_test)
    params = model.named_steps["model"].get_params()
    keep = {k: params[k] for k in ["n_estimators","max_depth","min_samples_split","min_samples_leaf","max_features"] if k in params}
    rows.append({
        "Model": name,
        "Train RMSE": rmse_tr, "Train MAE": mae_tr, "Train R²": r2_tr,
        "Test RMSE": rmse_te, "Test MAE": mae_te, "Test R²": r2_te,
        "Best Params": keep
    })

metrics_df = pd.DataFrame(rows)

# Feature importance
def get_feature_names_from_pipe(pipe):
    pre = pipe.named_steps["preprocess"]
    num_names = numeric_features
    cat_names = list(pre.named_transformers_["cat"].named_steps["onehot"].get_feature_names_out(categorical_features)) if len(categorical_features)>0 else []
    return np.array(list(num_names) + cat_names)

def feature_importance_df(pipe, top_k=20):
    model = pipe.named_steps["model"]
    feats = get_feature_names_from_pipe(pipe)
    imp = pd.DataFrame({"feature": feats, "importance": model.feature_importances_})
    return imp.sort_values("importance", ascending=False).head(top_k).reset_index(drop=True)

dt_imp = feature_importance_df(dt_best, 20)
rf_imp = feature_importance_df(rf_best, 20)

# Save & display
metrics_path = "tree_forest_metrics.csv"
dt_imp_path = "decision_tree_top_importances.csv"
rf_imp_path = "random_forest_top_importances.csv"

metrics_df.to_csv(metrics_path, index=False)
dt_imp.to_csv(dt_imp_path, index=False)
rf_imp.to_csv(rf_imp_path, index=False)

print("\n=== Tree & Forest Model Comparison ===")
print(metrics_df.round(4).to_string(index=False))

print("\n=== Decision Tree: Top 20 Feature Importances ===")
print(dt_imp.round(6).to_string(index=False))

print("\n=== Random Forest: Top 20 Feature Importances ===")
print(rf_imp.round(6).to_string(index=False))

# Charts
plt.figure()
plt.bar(metrics_df["Model"], metrics_df["Test R²"])
plt.title("Test R² (Decision Tree vs Random Forest)")
plt.xlabel("Model")
plt.ylabel("Test R²")
plt.tight_layout()
plt.savefig("trees_test_r2.png")
plt.close()

plt.figure()
plt.barh(rf_imp["feature"].iloc[::-1], rf_imp["importance"].iloc[::-1])
plt.title("Random Forest: Top 20 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.savefig("rf_top20_importances.png")
plt.close()


=== Tree & Forest Model Comparison ===
       Model  Train RMSE  Train MAE  Train R²  Test RMSE  Test MAE  Test R²                                                                                                   Best Params
DecisionTree      7.9264     6.7398    0.1577     8.0238    6.7431   0.1105                        {'max_depth': 4, 'min_samples_split': 10, 'min_samples_leaf': 2, 'max_features': None}
RandomForest      6.1753     5.3957    0.4888     8.0916    7.0362   0.0954 {'n_estimators': 250, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 2, 'max_features': 'sqrt'}

=== Decision Tree: Top 20 Feature Importances ===
             feature  importance
           Diagnosis    0.319872
                 ADL    0.266259
FunctionalAssessment    0.180577
  BehavioralProblems    0.071462
        SleepQuality    0.043144
    MemoryComplaints    0.036735
         DiastolicBP    0.034523
          SystolicBP    0.017825
         DietQuality    0.016159
    CholesterolTotal 